Chronos (next Nos. Prediction) on dataset of sales store
-

https://huggingface.co/amazon/chronos-2

https://www.kaggle.com/datasets/abhishekjaiswal4896/store-sales-dataset?resource=download

In [1]:
!pip install "chronos-forecasting>=2.0"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline
from sklearn.metrics import root_mean_squared_error
import os
os.chdir("C:/Users/dbda.STUDENTSDC/Desktop/Practical Machine Learning/Datasets")

c:\Users\dbda.STUDENTSDC\.conda\envs\hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")

In [4]:
df =pd.read_csv("store_sales.csv",parse_dates=["date"])
df.head()

,date,store,sales,promo,holiday
0,2022-01-01,1,184.78,0,0
1,2022-01-02,1,192.62,0,0
2,2022-01-03,1,212.68,0,0
3,2022-01-04,1,249.58,1,0
4,2022-01-05,1,223.50,0,0


In [5]:
df = df.drop(['holiday','promo'],axis=1)
df = df.rename(columns={"date":"timestamp","store_id":"store","sales_amount":"sales"})
df.head()

,timestamp,store,sales
0,2022-01-01,1,184.78
1,2022-01-02,1,192.62
2,2022-01-03,1,212.68
3,2022-01-04,1,249.58
4,2022-01-05,1,223.50


In [6]:
df_1 = df[df['store'] ==1].copy() # check for only store =1
train = df_1.iloc[:-12]
test = df_1.iloc[-12:]
train.shape, test.shape

((718, 3), (12, 3))

In [7]:
future_df = test.copy()
future_df['sales'] = None

In [8]:
pred_df = pipeline.predict_df(
    train,future_df = future_df,
    validate_inputs = False,
    prediction_length = 12, # nos. of steps to forecast => predict for next 12 days
    id_column = 'store', # column indentifying diff. time series
    timestamp_column ='timestamp', # column  with date time info
    target='sales'  # column(s) with time series values to predict
)
y_pred = pred_df['predictions']
pred_df

,store,timestamp,target_name,predictions,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
0,1,2023-12-20,sales,265.556244,254.216400,256.992157,259.523315,262.216278,265.556244,269.408752,274.596924,282.121368,294.891968
1,1,2023-12-21,sales,253.896042,242.462723,245.412689,247.995178,250.736771,253.896042,257.737823,262.646637,270.132538,282.354980
2,1,2023-12-22,sales,236.070435,224.855621,227.715286,230.281174,232.981750,236.070435,239.796356,244.465286,251.355438,262.704010
3,1,2023-12-23,sales,226.348572,215.960663,218.575958,220.904953,223.493851,226.348572,229.804459,234.235733,241.038010,252.249954
4,1,2023-12-24,sales,229.289917,218.213089,220.917938,223.369476,226.091095,229.289917,233.236572,238.188126,245.349091,256.459259
5,1,2023-12-25,sales,245.951202,234.508240,237.402786,239.990555,242.797424,245.951202,249.967697,254.992737,262.674866,274.528748
6,1,2023-12-26,sales,262.395996,250.919144,253.824402,256.529938,259.288208,262.395996,266.510315,271.760101,280.032104,292.485382
7,1,2023-12-27,sales,266.289642,254.481842,257.366486,260.133545,263.012970,266.289642,270.315765,275.461914,283.553802,296.329987
8,1,2023-12-28,sales,253.931946,242.752838,245.511795,248.064438,250.734894,253.931946,257.998627,263.089264,270.564178,282.267761
9,1,2023-12-29,sales,236.073730,225.425735,228.080200,230.507080,233.133087,236.073730,239.821075,244.662262,251.660782,262.442902


In [9]:
root_mean_squared_error(test['sales'], y_pred)

17.796664340317836

Finding the prediction for all stores for next 10 days

In [10]:
from itertools import product

In [12]:
date_values=pd.date_range(start='2024-01-01',periods=10,freq='MS').values
stores= np.arange(1,11)
cartesian = list(product(date_values,stores))
future_vals = pd.DataFrame(cartesian,columns=['timestamp', 'store'])
future_vals['sales'] = None
future_vals


,timestamp,store,sales
0,2024-01-01,1,None
1,2024-01-01,2,None
2,2024-01-01,3,None
3,2024-01-01,4,None
4,2024-01-01,5,None
...,...,...,...
95,2024-10-01,6,None
96,2024-10-01,7,None
97,2024-10-01,8,None
98,2024-10-01,9,None


In [16]:
# prediction_length should match the number of rows per store in future_df
pred_df = pipeline.predict_df(
    df,
    future_df=future_df,
    validate_inputs=False,
    prediction_length=10, # predict for 10 days
    quantile_levels=[0.025,0.975],
    id_column='store',     # This tells the model to loop through all stores
    timestamp_column='timestamp',
    target='sales'
)
pred_df

,store,timestamp,target_name,predictions,0.025,0.975
0,1,2024-01-01,sales,246.875244,230.851196,301.568634
1,1,2024-01-02,sales,262.520752,246.323563,321.490570
2,1,2024-01-03,sales,266.172241,249.465576,327.338562
3,1,2024-01-04,sales,253.744919,238.469086,310.409973
4,1,2024-01-05,sales,236.430511,221.391434,291.120941
...,...,...,...,...,...,...
95,10,2024-01-06,sales,210.542816,195.916107,266.938751
96,10,2024-01-07,sales,212.870575,198.317215,270.917480
97,10,2024-01-08,sales,227.735626,211.751419,282.346619
98,10,2024-01-09,sales,244.221146,227.441467,302.908264


In [ ]:
# 1. Create an empty list to store results
rmse_list = []

# 2. Get the Actual Values (Ground Truth)
# We take the last 10 days of the original dataframe for comparison
test_all = df.groupby('store').tail(10)

# 3. The Loop (Calculates and stores)
for s_id in test_all['store'].unique():
     # Filter actuals for this specific store
    actual = test_all[test_all['store'] == s_id]['sales']

    # Filter predictions for this specific store
    # Ensure we sort by timestamp so they align with 'actual'
    predicted = pred_df[pred_df['store'] == s_id].sort_values('timestamp')['predictions']

     # We use .values to ensure we are comparing the numbers, not the indices
    rmse_val = root_mean_squared_error(actual.values, predicted.values)
    
    # Store as a tuple (store_id, rmse_value)
    rmse_list.append((s_id, rmse_val))

# 4. Sort the list by the second element (the RMSE value)
rmse_list.sort(key=lambda x: x[1])

# 5. Print the sorted results
print(f"{'Store ID':<10} | {'RMSE':<10}")
print("-" * 25)
for s_id, rmse_val in rmse_list:
    print(f"Store {s_id:<4} | {rmse_val:.4f}")

Store ID   | RMSE      
-------------------------
Store 4    | 21.0178
Store 10   | 25.0899
Store 6    | 25.9046
Store 5    | 26.6346
Store 9    | 32.3058
Store 2    | 33.8072
Store 1    | 35.0380
Store 3    | 35.0778
Store 7    | 35.7812
Store 8    | 36.8903
